In [22]:
import os, json, time, glob, requests, pandas as pd, numpy as np
from collections import Counter
from kaggle_secrets import UserSecretsClient


# Kaggle:
secret_label = "OpenAlex"
secret_value = UserSecretsClient().get_secret(secret_label)
# Jupyter trên máy: from getpass import getpass; KEY = getpass("OpenAlex key: ")

RAW = "/kaggle/working/data/raw"            # Jupyter: "data/raw"
os.makedirs(f"{RAW}/works", exist_ok=True)
print("Đã có key:", bool(secret_value))

Đã có key: True


In [23]:
def get(path, params, retries=4):
    params = {**params, "api_key": secret_value}      # tên tham số BẮT BUỘC là api_key
    for i in range(retries):
        try:
            r = requests.get(f"https://api.openalex.org/{path}", params=params, timeout=60)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i); continue
            if r.status_code >= 400:                   # in lý do lỗi từ máy chủ để dễ sửa
                raise requests.HTTPError(f"{r.status_code}: {r.text[:300]}")
            return r.json()
        except requests.ConnectionError:
            time.sleep(2 ** i)
    raise RuntimeError(f"Thất bại sau {retries} lần: {path}")

def count_works(filter_str):
    m = get("works", {"filter": filter_str, "per_page": 1})["meta"]
    return m["count"], m.get("cost_usd"), m["x_query"]["oql"]

count_works("title_and_abstract.search:artificial intelligence higher education")   # kiểm tra nhanh

(33952,
 0.001,
 'get works where title-abstract has (artificial intelligence higher education)')

In [24]:
S = "title_and_abstract.search:artificial intelligence higher education"
T = ",type:article|review|conference-paper"
variants = {
    "A – từ khoá rời":             S,
    "B' – A + loại tài liệu":     S + T,
    "C – article + tiếng Anh":     S + ",type:article,language:en",
    "G – cụm từ chính xác":      'title_and_abstract.search:"artificial intelligence" "higher education"' + T,
    "F – topic + (HE OR university)": 'primary_topic.id:T14414,title_and_abstract.search:"higher education" OR university' + T,
}
rows = []
for name, f in variants.items():
    try:
        n, cost, oql = count_works(f)
        rows.append({"biến thể": name, "count": n, "OpenAlex hiểu là": oql})
    except Exception as e:
        rows.append({"biến thể": name, "count": None, "OpenAlex hiểu là": f"LỖI: {e}"})
    time.sleep(0.3)
variant_df = pd.DataFrame(rows)
variant_df["% so với A"] = (variant_df["count"] / variant_df.loc[0, "count"] * 100).round(1)
variant_df

,biến thể,count,OpenAlex hiểu là,% so với A
0,A – từ khoá rời,33952,get works where title-abstract has (artificial...,100.0
1,B' – A + loại tài liệu,27551,get works where title-abstract has (artificial...,81.1
2,C – article + tiếng Anh,15120,get works where language is [English](en)\n a...,44.5
3,G – cụm từ chính xác,21284,"get works where title-abstract has (stemmed ""a...",62.7
4,F – topic + (HE OR university),23870,get works where topic is (T14414)\n and title...,70.3


In [43]:
N_SAMPLE = 100                                        # cỡ mẫu: 20 để chọn nhanh, 100 để có con số đáng tin
candidates = {"G": variants["G – cụm từ chính xác"], "F": variants["F – topic + (HE OR university)"]}
samples = {}
for name, f in candidates.items():
    d = get("works", {"filter": f, "sample": N_SAMPLE, "seed": 1, "per_page": N_SAMPLE,
                      "select": "id,title,publication_year,type"})
    samples[name] = pd.DataFrame(d["results"])
    time.sleep(0.3)
print({k: len(v) for k, v in samples.items()})        # phải là {'G': 100, 'F': 100}

{'G': 100, 'F': 100}


In [44]:
for name, df in samples.items():
    print(f"\n===== {name} =====")
    for i, row in df.iterrows():
        print(f"{i:2d}. [{row['publication_year']}] {row['title']}")


===== G =====
 0. [2026] The impact of university students' AI attitudes on AI-assisted creativity: the mediating role of AI usage motivation and the moderating role of AI dependency
 1. [2026] Artificial Intelligence in Personalized Learning and Student Assessment: Challenges and Implementation Solutions
 2. [2026] THE IMPACT OF APPLYING GEN-AI TO KNOWLEDGE RETENTION AMONG STUDENTS IN EDUCATIONAL INSTITUTES: A SCOPING REVIEW
 3. [2026] Exploring success factors that impact artificial intelligence adoption on technological drivers and managerial capabilities
 4. [2026] Generative Artificial Intelligence (GenAI) in Higher Education from Prospective Teachers’ Perspectives: A Qualitative Study on Support Opportunities and Usage Challenges
 5. [2026] "Beyond Digital Literacy: Developing AI Readiness and Critical Thinking Skills among Senior Secondary Students"
 6. [2026] THEORETICAL AND METHODOLOGICAL FOUNDATIONS FOR DEVELOPING THE PROFESSIONAL-METHODOLOGICAL COMPETENCE OF PROSPECTIVE ENG

In [48]:
# Số thứ tự (cột bên trái khi in tiêu đề ở ô trên) các bài LẠC ĐỀ. Mọi bài còn lại được tính là đúng chủ đề.
zeros = {
    # G: AI trong doanh nghiệp (3), phổ thông (5), kinh tế không gian (8), AI–5G với giáo dục nói chung (22),
    #    giáo dục đại học nhưng KHÔNG về AI (18, 20, 24, 29, 43, 54, 62, 63, 64, 67, 73, 82, 85, 88, 96),
    #    tổng quan hội nghị chung (51), xếp hạng "trí tuệ" của trường bằng fuzzy VIKOR (91)
    "G": [3, 5, 8, 18, 20, 22, 24, 29, 43, 51, 54, 62, 63, 64, 67, 73, 82, 85, 88, 91, 96],
    # F: phổ thông (12), AI–5G nói chung (24), không về AI (30), công cụ cho giáo viên phổ thông Magicschool.ai (44)
    "F": [12, 24, 30, 44],
}
result = {}
for name, df in samples.items():
    df["đúng"] = 1
    df.loc[zeros[name], "đúng"] = 0
    k, p = len(df), df["đúng"].mean()
    margin = 1.96 * np.sqrt(p * (1 - p) / k)          # sai số ~95%
    result[name] = f"{df['đúng'].sum()}/{k} = {p:.0%} ± {margin:.0%}"
result

{'G': '79/100 = 79% ± 8%', 'F': '96/100 = 96% ± 4%'}

In [47]:
F = ('primary_topic.id:T14414,title_and_abstract.search:"higher education" OR university'
     ',type:article|review|conference-paper,publication_year:2015-2026')
count_works(F)

(23712,
 0.001,
 'get works where topic is (T14414)\n  and published from 2015 through 2026\n  and title-abstract has (stemmed "higher education" OR university)\n  and type is ([article](article) or [conference-paper](conference-paper) or [review](review))')

In [31]:
FIELDS = ("id,doi,title,publication_year,publication_date,type,language,cited_by_count,fwci,"
          "authorships,primary_topic,keywords,abstract_inverted_index,open_access,referenced_works_count")
d = get("works", {"filter": F, "select": FIELDS, "per_page": 200})
n, cost = d["meta"]["count"], d["meta"]["cost_usd"]
print(f"{n} bài | {cost} USD/trang | ước tính: {np.ceil(n / 200) * cost:.2f} USD")

23712 bài | 0.001 USD/trang | ước tính: 0.12 USD


In [32]:
STATE = f"{RAW}/cursor_state.json"
state = json.load(open(STATE)) if os.path.exists(STATE) else {"cursor": "*", "page": 0}

while state["cursor"]:
    d = get("works", {"filter": F, "select": FIELDS, "per_page": 200, "cursor": state["cursor"]})
    with open(f"{RAW}/works/page_{state['page']:04d}.json", "w", encoding="utf-8") as fh:
        json.dump(d["results"], fh, ensure_ascii=False)
    state = {"cursor": d["meta"]["next_cursor"], "page": state["page"] + 1}
    json.dump(state, open(STATE, "w"))
    if state["page"] % 20 == 0:
        print("đã tải", state["page"], "trang")
    time.sleep(0.2)
print("Xong:", state["page"], "trang")

đã tải 20 trang
đã tải 40 trang
đã tải 60 trang
đã tải 80 trang
đã tải 100 trang
Xong: 119 trang


In [33]:
yearly = get("works", {"filter": F, "group_by": "publication_year"})["group_by"]
json.dump(yearly, open(f"{RAW}/yearly.json", "w"), ensure_ascii=False)

F_NOYEAR = F.replace(",publication_year:2015-2026", "")
rows = []
for y in range(2015, 2027):
    d = get("works", {"filter": f"{F_NOYEAR},publication_year:{y}", "group_by": "authorships.countries"})
    rows += [{"year": y, "country": g["key"].split("/")[-1], "name": g["key_display_name"],
              "count": g["count"]} for g in d["group_by"]]
    time.sleep(0.2)
json.dump(rows, open(f"{RAW}/country_year.json", "w", encoding="utf-8"), ensure_ascii=False)
pd.DataFrame(yearly)[["key_display_name", "count"]].sort_values("key_display_name")

,key_display_name,count
11,2015,3
10,2016,9
9,2017,14
8,2018,36
7,2019,78
6,2020,130
5,2021,196
4,2022,218
3,2023,1089
2,2024,3686


In [34]:
works_raw = [w for f in sorted(glob.glob(f"{RAW}/works/*.json")) for w in json.load(open(f, encoding="utf-8"))]
inst_count = Counter(i["id"] for w in works_raw for a in w["authorships"] for i in a.get("institutions", []))
top_ids = [i.split("/")[-1] for i, _ in inst_count.most_common(400)]

insts = []
for k in range(0, len(top_ids), 50):                 # mỗi lần hỏi 50 trường
    d = get("institutions", {"filter": "openalex:" + "|".join(top_ids[k:k+50]), "per_page": 50,
                             "select": "id,display_name,country_code,type,geo,works_count,cited_by_count"})
    insts += d["results"]
    time.sleep(0.2)
json.dump(insts, open(f"{RAW}/institutions.json", "w", encoding="utf-8"), ensure_ascii=False)
len(insts)                                           # phải là 400

400

In [35]:
inst_df = pd.json_normalize(insts)
inst_df["id"] = inst_df["id"].str.split("/").str[-1]
inst_df["so_bai_trong_du_lieu"] = inst_df["id"].map({k.split("/")[-1]: v for k, v in inst_count.items()})
print("Đủ ID:", set(inst_df["id"]) == set(top_ids))
print(inst_df["type"].value_counts())
inst_df[inst_df["country_code"] == "VN"][["display_name", "so_bai_trong_du_lieu"]]

Đủ ID: True
type
education     398
government      1
facility        1
Name: count, dtype: int64


,display_name,so_bai_trong_du_lieu
2,FPT University,80
120,"Vietnam National University, Hanoi",51
286,VNU University of Education,31
317,Industrial University of Ho Chi Minh City,27
330,Van Lang University,26


In [49]:
# ---- 1. Đọc lại dữ liệu thô và khử trùng ID (quy tắc 3) ----
works_all = [w for f in sorted(glob.glob(f"{RAW}/works/*.json")) for w in json.load(open(f, encoding="utf-8"))]
seen, works_raw = set(), []
for w in works_all:
    if w["id"] not in seen:
        seen.add(w["id"]); works_raw.append(w)
dup_id_count = len(works_all) - len(works_raw)
print(f"Đã tải: {len(works_all)} dòng | trùng ID bị bỏ: {dup_id_count} | còn lại: {len(works_raw)} bài")

n = len(works_raw)
pct = lambda k: round(100 * k / n, 1)
countries_of = lambda w: {c for a in w.get("authorships", []) for c in (a.get("countries") or [])}

# ---- 2. Chuẩn bị dữ liệu cho từng quy tắc ----
dois  = [w["doi"] for w in works_raw if w.get("doi")]
years = pd.Series([w.get("publication_year") for w in works_raw])
cites = pd.Series([w.get("cited_by_count") for w in works_raw], dtype="float")
types = Counter(w.get("type") for w in works_raw)
langs = Counter(w.get("language") for w in works_raw)
api_count = count_works(F)[0]                       # F phải là ĐÚNG filter đã dùng khi tải (ô B5)
q1, q3 = cites.quantile([0.25, 0.75]); upper = q3 + 1.5 * (q3 - q1)
PRECISION = round(100 * samples["F"]["đúng"].mean(), 1)   # lấy thẳng từ kết quả chấm 100 bài ở B4
N_SAMPLE_F = len(samples["F"])

def row(rule, metric, value, text, ok=None):
    return {"quy tắc": rule, "chỉ số": metric, "giá trị": value, "ngưỡng": text,
            "kết quả": "—" if ok is None else ("ĐẠT" if ok(value) else "XEM LẠI")}

# ---- 3. Bảng chấm điểm theo 7 quy tắc ----
scorecard = pd.DataFrame([
    row("1. Liên quan", f"Precision mẫu {N_SAMPLE_F} bài (%)", PRECISION, ">= 70", lambda v: v >= 70),
    row("2. Đầy đủ", "Thiếu tóm tắt (%)", pct(sum(w.get("abstract_inverted_index") is None for w in works_raw)), "<= 40", lambda v: v <= 40),
    row("2. Đầy đủ", "Không có quốc gia (%)", pct(sum(not countries_of(w) for w in works_raw)), "<= 30", lambda v: v <= 30),
    row("2. Đầy đủ", "Thiếu primary_topic (%)", pct(sum(w.get("primary_topic") is None for w in works_raw)), "= 0", lambda v: v == 0),
    row("2. Đầy đủ", "Thiếu DOI (%)", pct(n - len(dois)), "báo cáo"),
    row("3. Duy nhất", "Trùng ID khi tải (dòng, đã loại)", dup_id_count, "báo cáo"),
    row("3. Duy nhất", "Trùng DOI, khác ID (bài)", len(dois) - len(set(dois)), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Năm ngoài 2015–2026 (bài)", int((~years.between(2015, 2026)).sum()), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Loại ngoài article/review/conference (bài)",
        sum(v for k, v in types.items() if k not in {"article", "review", "conference-paper"}), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Trích dẫn âm hoặc thiếu (bài)", int((cites.isna() | (cites < 0)).sum()), "= 0", lambda v: v == 0),
    row("5. Nhất quán", "Lệch số bài (sau khử trùng) so với API (%)", round(100 * abs(n - api_count) / api_count, 2), "<= 1", lambda v: v <= 1),
    row("6. Đại diện", "Không phải tiếng Anh (%)", pct(n - langs.get("en", 0)), "báo cáo"),
    row("6. Đại diện", "Bài năm 2026, chưa đủ năm (%)", pct(int((years == 2026).sum())), "báo cáo"),
    row("7. Ngoại lai", f"Trích dẫn > {upper:.0f} (Q3 + 1.5·IQR) (%)", pct(int((cites > upper).sum())), "báo cáo"),
])
print("Số bài dùng để chấm:", n, "| API:", api_count)
scorecard

Đã tải: 23699 dòng | trùng ID bị bỏ: 154 | còn lại: 23545 bài
Số bài dùng để chấm: 23545 | API: 23712


,quy tắc,chỉ số,giá trị,ngưỡng,kết quả
0,1. Liên quan,Precision mẫu 100 bài (%),96.0,>= 70,ĐẠT
1,2. Đầy đủ,Thiếu tóm tắt (%),10.4,<= 40,ĐẠT
2,2. Đầy đủ,Không có quốc gia (%),26.6,<= 30,ĐẠT
3,2. Đầy đủ,Thiếu primary_topic (%),0.0,= 0,ĐẠT
4,2. Đầy đủ,Thiếu DOI (%),10.2,báo cáo,—
5,3. Duy nhất,"Trùng ID khi tải (dòng, đã loại)",154.0,báo cáo,—
6,3. Duy nhất,"Trùng DOI, khác ID (bài)",5.0,= 0,XEM LẠI
7,4. Hợp lệ,Năm ngoài 2015–2026 (bài),0.0,= 0,ĐẠT
8,4. Hợp lệ,Loại ngoài article/review/conference (bài),0.0,= 0,ĐẠT
9,4. Hợp lệ,Trích dẫn âm hoặc thiếu (bài),0.0,= 0,ĐẠT


In [50]:
miss = pd.DataFrame({
    "year": years,
    "thiếu_tóm_tắt": [w.get("abstract_inverted_index") is None for w in works_raw],
    "thiếu_quốc_gia": [not countries_of(w) for w in works_raw],
    "tiếng_Anh": [w.get("language") == "en" for w in works_raw],
})
by_year = (miss.groupby("year")[["thiếu_tóm_tắt", "thiếu_quốc_gia"]].mean() * 100).round(1)
by_year["số bài"] = miss.groupby("year").size()
by_lang = (miss.groupby("tiếng_Anh")[["thiếu_tóm_tắt", "thiếu_quốc_gia"]].mean() * 100).round(1)
display(by_year)          # Jupyter chỉ tự hiển thị dòng cuối, nên dùng display() cho cả hai bảng
display(by_lang)

,thiếu_tóm_tắt,thiếu_quốc_gia,số bài
year,,,
2015,100.0,100.0,3
2016,22.2,55.6,9
2017,28.6,50.0,14
2018,13.9,38.9,36
2019,32.1,35.9,78
2020,13.1,33.8,130
2021,16.4,32.8,195
2022,11.6,24.5,216
2023,7.0,23.5,1085


,thiếu_tóm_tắt,thiếu_quốc_gia
tiếng_Anh,,
False,8.2,33.3
True,11.6,22.8


In [51]:
top = sorted(works_raw, key=lambda w: w.get("cited_by_count") or 0, reverse=True)[:10]
pd.DataFrame([{"title": w["title"], "year": w["publication_year"], "type": w["type"],
               "cites": w["cited_by_count"], "fwci": w.get("fwci")} for w in top])

,title,year,type,cites,fwci
0,Systematic review of research on artificial in...,2019,review,7126,20.8637
1,Students’ voices on generative AI: perceptions...,2023,article,2463,572.5626
2,Exploring the impact of artificial intelligenc...,2017,article,2009,74.5450
3,What Is the Impact of ChatGPT on Education? A ...,2023,article,1998,465.1199
4,Artificial intelligence in higher education: t...,2023,article,1957,455.1198
5,ChatGPT: Bullshit spewer or the end of traditi...,2023,article,1803,419.5351
6,A comprehensive AI policy education framework ...,2023,article,1556,361.3982
7,Intelligence Unleashed: An argument for AI in ...,2016,article,1410,50.2635
8,Embracing the future of Artificial Intelligenc...,2024,article,1099,413.8091
9,New Era of Artificial Intelligence in Educatio...,2023,article,1080,251.3955


In [52]:
# D5 – Biến kết quả chấm điểm thành quyết định làm sạch cho Ngày 3
ACTIONS = {
    "Precision":              "Quay lại Phần B, thu hẹp truy vấn rồi tải lại",
    "Thiếu tóm tắt":          "Giữ dòng; phân tích từ khoá bằng cột keywords hoặc chỉ trên bài có tóm tắt (pairwise deletion)",
    "Không có quốc gia":      "Giữ dòng cho xu hướng; chỉ loại khỏi bản đồ ⑨ và network ⑩; ghi tỷ lệ vào phần giới hạn",
    "Thiếu primary_topic":    "Kiểm tra lại truy vấn F (lẽ ra phải luôn có topic)",
    "Thiếu DOI":              "Không xử lý; kiểm tra trùng chỉ dựa trên ID và DOI có sẵn",
    "Trùng ID":               "Đã loại ngay khi đọc dữ liệu (giữ bản đầu tiên)",
    "Trùng DOI":              "drop_duplicates theo DOI, giữ bản có nhiều trích dẫn hơn",
    "Năm ngoài":              "Lọc bỏ dòng ngoài 2015–2026",
    "Loại ngoài":             "Lọc bỏ loại tài liệu ngoài article/review/conference-paper",
    "Trích dẫn âm":           "Loại khỏi biểu đồ tác động ⑤⑥",
    "Lệch số bài":            "Nếu > 1%: xoá cursor_state.json và thư mục works/, tải lại C2",
    "Không phải tiếng Anh":   "Giữ tất cả; chỉ lọc tiếng Anh khi phân tích từ khoá ⑫ và ghi rõ",
    "Bài năm 2026":           "Loại 2026 khỏi YoY/CAGR, hoặc vẽ nét đứt và chú thích 'chưa đủ năm'",
    "Trích dẫn >":            "Giữ (ngoại lai thật, xem D4); dùng log1p và so sánh bằng fwci",
}
def action_for(metric):
    return next((a for k, a in ACTIONS.items() if metric.startswith(k)), "")

plan = scorecard.assign(**{"hành động (Ngày 3)": scorecard["chỉ số"].map(action_for)})
plan = plan[plan["kết quả"] != "ĐẠT"][["quy tắc", "chỉ số", "giá trị", "kết quả", "hành động (Ngày 3)"]]
pd.set_option("display.max_colwidth", 140)
plan

,quy tắc,chỉ số,giá trị,kết quả,hành động (Ngày 3)
4,2. Đầy đủ,Thiếu DOI (%),10.2,—,Không xử lý; kiểm tra trùng chỉ dựa trên ID và DOI có sẵn
5,3. Duy nhất,"Trùng ID khi tải (dòng, đã loại)",154.0,—,Đã loại ngay khi đọc dữ liệu (giữ bản đầu tiên)
6,3. Duy nhất,"Trùng DOI, khác ID (bài)",5.0,XEM LẠI,"drop_duplicates theo DOI, giữ bản có nhiều trích dẫn hơn"
11,6. Đại diện,Không phải tiếng Anh (%),35.7,—,Giữ tất cả; chỉ lọc tiếng Anh khi phân tích từ khoá ⑫ và ghi rõ
12,6. Đại diện,"Bài năm 2026, chưa đủ năm (%)",42.8,—,"Loại 2026 khỏi YoY/CAGR, hoặc vẽ nét đứt và chú thích 'chưa đủ năm'"
13,7. Ngoại lai,Trích dẫn > 5 (Q3 + 1.5·IQR) (%),13.9,—,"Giữ (ngoại lai thật, xem D4); dùng log1p và so sánh bằng fwci"


In [53]:
import shutil
log = {"ngay_tai": time.strftime("%Y-%m-%d"), "filter": F, "count_api": api_count,
       "so_dong_tai_ve": len(works_all), "trung_id_da_loai": dup_id_count, "so_bai": n,
       "so_trang": len(glob.glob(f"{RAW}/works/*.json")), "so_truong": len(insts),
       "precision_F": PRECISION, "co_mau": N_SAMPLE_F}
json.dump(log, open(f"{RAW}/download_log.json", "w"), ensure_ascii=False, indent=2)
variant_df.to_csv(f"{RAW}/query_variants.csv", index=False)
scorecard.to_csv(f"{RAW}/quality_scorecard.csv", index=False)
plan.to_csv(f"{RAW}/cleaning_plan.csv", index=False)
pd.concat([df.assign(bien_the=k) for k, df in samples.items()]).to_csv(f"{RAW}/precision_samples.csv", index=False)
shutil.make_archive("/kaggle/working/raw_openalex", "zip", RAW)   # tải file này ở panel Output bên phải
log

{'ngay_tai': '2026-10-23',
 'filter': 'primary_topic.id:T14414,title_and_abstract.search:"higher education" OR university,type:article|review|conference-paper,publication_year:2015-2026',
 'count_api': 23712,
 'so_dong_tai_ve': 23699,
 'trung_id_da_loai': 154,
 'so_bai': 23545,
 'so_trang': 119,
 'so_truong': 400,
 'precision_F': np.float64(96.0),
 'co_mau': 100}